# Feature engineering: fixing what `02_baseline.ipynb` flagged as naive

`02_baseline.ipynb` ended with a list of four things it did on purpose but
naively: blanket median/`'None'` imputation, an untuned `Ridge(alpha=10)`, no
feature engineering, and one-hot encoding ordinal quality columns instead of
respecting their order. This notebook works through that list, **one change
at a time**, re-running the exact same 5-fold CV setup after each change --
so we know which changes actually helped before spending a Kaggle submission,
rather than bundling everything together and guessing which part mattered.

Spoiler, stated up front so it doesn't read as a failure later: most of these
"obviously correct" fixes barely move the cross-validated score for a
**linear, regularized** model like Ridge. That's not a sign anything is
wrong -- it's a real property of linear models worth understanding, and it's
exactly why we check each step instead of assuming.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import KFold, cross_val_score

train = pd.read_csv('../inputs/train.csv')
test = pd.read_csv('../inputs/test.csv')
train = train[~train['Id'].isin([524, 1299])].reset_index(drop=True)

y_train = np.log1p(train['SalePrice'])
test_ids = test['Id']
train_features = train.drop(columns=['Id', 'SalePrice'])
test_features = test.drop(columns=['Id'])
n_train = len(train_features)
combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)


def cv_score(X, y, alpha=10):
    # 5-fold CV log-RMSE for Ridge -- the same measuring stick for every step below.
    scores = cross_val_score(Ridge(alpha=alpha), X, y, cv=kf, scoring='neg_root_mean_squared_error')
    rmse = -scores
    return rmse.mean(), rmse.std()


results = {}  # step name -> (mean_rmse, std_rmse, n_features)

## Step 0: reproduce the baseline, as our anchor

Before changing anything, we rebuild `02_baseline.ipynb`'s exact preprocessing
in this notebook (blanket median-impute numerics, fill categoricals with
`'None'`, one-hot encode) and record its CV score. Every later step gets
compared back to this one number.

In [2]:
numeric_cols = combined.select_dtypes(include=[np.number]).columns
categorical_cols = combined.select_dtypes(exclude=[np.number]).columns

step0 = combined.copy()
train_medians = step0.loc[:n_train - 1, numeric_cols].median()
step0[numeric_cols] = step0[numeric_cols].fillna(train_medians)
step0[categorical_cols] = step0[categorical_cols].fillna('None')
step0_encoded = pd.get_dummies(step0, columns=list(categorical_cols), drop_first=True)

X0 = step0_encoded.iloc[:n_train].reset_index(drop=True)
mean0, std0 = cv_score(X0, y_train)
results['0. baseline (repro)'] = (mean0, std0, X0.shape[1])
print(f'Step 0 -- baseline reproduction:  log-RMSE {mean0:.4f} (+/- {std0:.4f})  [{X0.shape[1]} features]')

Step 0 -- baseline reproduction:  log-RMSE 0.1154 (+/- 0.0091)  [266 features]


## Step 1: semantic-aware missing value handling

`01_eda.ipynb` already flagged the key quirk: for most categorical columns,
`NaN` doesn't mean "unknown" -- it means "this house doesn't have the
feature" (no pool, no basement, no garage). The baseline's blanket rule
(`fillna('None')` for every categorical, `fillna(median)` for every numeric)
gets the categorical half right by accident, but gets two things wrong:

1. **Numeric absent-feature columns** (`GarageArea`, `GarageCars`,
   `BsmtFullBath`, `BsmtHalfBath`, `BsmtFinSF1/2`, `BsmtUnfSF`,
   `TotalBsmtSF`, `MasVnrArea`) were median-filled -- e.g. a house with no
   basement got the *typical* basement square footage instead of `0`. These
   should be `0`, matching what "doesn't have this feature" actually means
   numerically. `GarageYrBlt` is the one exception: `0` would mean "the
   garage was built in year 0", which is nonsense for a linear model to
   ingest. We fill it with the house's own `YearBuilt` instead -- a
   reasonable assumption (garages are usually built with the house) that at
   least keeps the value in a sane range.
2. **A handful of categorical columns are genuinely missing, not absent**
   (`MSZoning`, `Functional`, `Utilities`, `Exterior1st`, `Exterior2nd`,
   `Electrical`, `KitchenQual`, `SaleType` -- 1-4 rows each, all in the test
   set). `data_description.txt` doesn't list `NaN`/`'None'` as a valid level
   for any of these -- every house *has* a zoning classification and a
   kitchen quality rating, we just don't know it for a few rows. Filling
   these with the literal string `'None'` (as the baseline's blanket rule
   does) invents a fake category that doesn't exist in the data dictionary.
   The defensible fill here is the column's **mode** (most common value).

One more genuinely-missing column, `LotFrontage` (486 missing, the largest
count of any of these), gets a better fill than a single global median:
**per-neighborhood median**, since lot frontage is strongly a function of
location (a well-known, effective trick specific to this dataset). As
always, every fill statistic (medians, modes) is computed from **training
rows only**, even though it gets applied to the combined train+test frame.

In [3]:
cat_absent = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType',
              'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond',
              'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
num_absent_zero = ['GarageArea', 'GarageCars', 'BsmtFullBath', 'BsmtHalfBath',
                   'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'MasVnrArea']
cat_truly_missing = ['MSZoning', 'Functional', 'Utilities', 'Exterior1st', 'Exterior2nd',
                      'Electrical', 'KitchenQual', 'SaleType']

step1 = combined.copy()
step1[cat_absent] = step1[cat_absent].fillna('None')
step1[num_absent_zero] = step1[num_absent_zero].fillna(0)
step1['GarageYrBlt'] = step1['GarageYrBlt'].fillna(step1['YearBuilt'])

train_rows = step1.iloc[:n_train]
neighborhood_frontage_median = train_rows.groupby('Neighborhood')['LotFrontage'].median()
step1['LotFrontage'] = step1['LotFrontage'].fillna(step1['Neighborhood'].map(neighborhood_frontage_median))

for col in cat_truly_missing:
    mode_value = train_rows[col].mode()[0]
    step1[col] = step1[col].fillna(mode_value)

remaining_na = step1.isna().sum()
print('Any columns still missing values?', dict(remaining_na[remaining_na > 0]) or 'none')

categorical_cols_1 = step1.select_dtypes(exclude=[np.number]).columns
step1_encoded = pd.get_dummies(step1, columns=list(categorical_cols_1), drop_first=True)
X1 = step1_encoded.iloc[:n_train].reset_index(drop=True)
mean1, std1 = cv_score(X1, y_train)
results['1. semantic missing-value fix'] = (mean1, std1, X1.shape[1])
print(f'Step 1 -- semantic missing-value fix:  log-RMSE {mean1:.4f} (+/- {std1:.4f})  [{X1.shape[1]} features]')

Any columns still missing values? none
Step 1 -- semantic missing-value fix:  log-RMSE 0.1156 (+/- 0.0092)  [258 features]


**Reading it:** log-RMSE goes from **0.1154 to 0.1156** -- statistically
nothing (well within the ~0.009 fold-to-fold noise), if anything a hair
worse. That's a real result, not a mistake: every column we "fixed" affects
at most a few hundred of ~2900 rows, and Ridge's regularization was already
shrinking those dummy/coefficient contributions toward zero regardless of
whether the fill value was semantically right or a generic median. **This
step is still worth doing** -- it removes fabricated values (a fake "typical"
garage year, a fake average basement size) that would actively mislead a
model that treats numeric magnitude literally, like a tree-based model
splitting on `GarageYrBlt < 1965`. It just isn't where a *linear* model's
score is won or lost.

## Step 2: ordinal-encode the quality/condition columns

Several columns share the exact same `Po < Fa < TA < Gd < Ex` scale (and a
few others have their own natural order) -- see `data_description.txt`. The
baseline one-hot encodes all of them, which throws the ordering away: it lets
`ExterQual == 'Gd'` and `ExterQual == 'Ex'` each get an independent
coefficient with no relationship to each other, when we know `Ex` is
strictly better than `Gd`, which is strictly better than `TA`. Mapping these
to integers (`0` for "doesn't exist", increasing with quality) lets a single
coefficient capture "each step up this scale is worth about this much",
instead of spending 4-6 independent coefficients to reconstruct the same
one-directional idea.

Deliberately **not** ordinal-encoded: `Fence` and `MiscFeature`. Their levels
mix genuinely different kinds of things (`GdPrv` = good *privacy*, `GdWo` =
good *wood* -- material and purpose, not one scale), so forcing a single
order onto them would be inventing structure that isn't really there. They
stay one-hot.

In [4]:
qual5 = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ordinal_cols_5level = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                       'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

ordinal_maps = {col: qual5 for col in ordinal_cols_5level}
ordinal_maps.update({
    'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'LotShape': {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3},
    'LandSlope': {'Gtl': 0, 'Mod': 1, 'Sev': 2},
    'Utilities': {'ELO': 0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3},
    'CentralAir': {'N': 0, 'Y': 1},
    'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
    'Functional': {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7},
})

step2 = step1.copy()
for col, mapping in ordinal_maps.items():
    step2[col] = step2[col].map(mapping)

unmapped = step2[list(ordinal_maps)].isna().sum()
print('Unmapped values introduced (should be all zero -- flags a typo in a mapping dict):')
print(unmapped[unmapped > 0].to_dict() or 'none')

categorical_cols_2 = step2.select_dtypes(exclude=[np.number]).columns
print(f'\n{len(ordinal_maps)} columns converted to ordinal integers; {len(categorical_cols_2)} true nominal columns remain one-hot encoded')

step2_encoded = pd.get_dummies(step2, columns=list(categorical_cols_2), drop_first=True)
X2 = step2_encoded.iloc[:n_train].reset_index(drop=True)
mean2, std2 = cv_score(X2, y_train)
results['2. + ordinal encoding'] = (mean2, std2, X2.shape[1])
print(f'\nStep 2 -- ordinal encoding:  log-RMSE {mean2:.4f} (+/- {std2:.4f})  [{X2.shape[1]} features]')

Unmapped values introduced (should be all zero -- flags a typo in a mapping dict):
none

20 columns converted to ordinal integers; 23 true nominal columns remain one-hot encoded



Step 2 -- ordinal encoding:  log-RMSE 0.1155 (+/- 0.0084)  [204 features]


**Reading it:** log-RMSE is essentially unchanged again (0.1155 vs. 0.1154),
but two things improved quietly: the fold-to-fold standard deviation dropped
(0.0084 vs. 0.0092) and we did it with **62 fewer features** (204 vs. 266) --
20 ordinal columns that used to cost 4-6 dummy columns each now cost exactly
1 column each. A model that's just as accurate with fewer, more meaningful
parameters is a real improvement even when the headline CV number doesn't
move -- it's a more stable, more interpretable model, which matters more the
smaller your training set is (we have 1458 rows).

## Step 3: engineered features -- and a genuinely important gotcha

Now the classic Ames feature-engineering additions: total square footage,
house age at sale, total bathrooms, total porch area, and a few "has this
feature at all" flags.

**Before running it, a prediction worth stating up front:** several of these
new features are just a *sum of columns that are already in the model*
(`TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF`; `HouseAge = YrSold -
YearBuilt`). For a linear model, that sum contains **zero new information**
-- Ridge can already learn any linear combination of `TotalBsmtSF`,
`1stFlrSF`, and `2ndFlrSF` by giving each its own coefficient; adding their
sum as a fourth column doesn't let it represent anything it couldn't
represent before. (This is very different from a tree-based model, which
splits on one column at a time and genuinely struggles to reconstruct "the
sum of three columns" from individual thresholds -- there, this kind of
feature engineering matters a lot.) We'll add the features *alongside* the
raw columns first, to confirm that prediction, then try it the other way.

In [5]:
step3 = step2.copy()
step3['TotalSF'] = step3['TotalBsmtSF'] + step3['1stFlrSF'] + step3['2ndFlrSF']
step3['HouseAge'] = (step3['YrSold'] - step3['YearBuilt']).clip(lower=0)
step3['RemodAge'] = (step3['YrSold'] - step3['YearRemodAdd']).clip(lower=0)
step3['TotalBath'] = step3['FullBath'] + 0.5 * step3['HalfBath'] + step3['BsmtFullBath'] + 0.5 * step3['BsmtHalfBath']
step3['TotalPorchSF'] = (step3['OpenPorchSF'] + step3['EnclosedPorch'] + step3['3SsnPorch']
                          + step3['ScreenPorch'] + step3['WoodDeckSF'])
step3['HasPool'] = (step3['PoolArea'] > 0).astype(int)
step3['HasGarage'] = (step3['GarageArea'] > 0).astype(int)
step3['HasFireplace'] = (step3['Fireplaces'] > 0).astype(int)
step3['Has2ndFloor'] = (step3['2ndFlrSF'] > 0).astype(int)
step3['HasBsmt'] = (step3['TotalBsmtSF'] > 0).astype(int)
# HouseAge/RemodAge are clipped at 0 -- a couple of rows record YearBuilt or
# YearRemodAdd as the same year as YrSold but with what looks like a
# same-year rounding quirk, producing a small negative "age" otherwise.

categorical_cols_3 = step3.select_dtypes(exclude=[np.number]).columns
step3_encoded = pd.get_dummies(step3, columns=list(categorical_cols_3), drop_first=True)
X3 = step3_encoded.iloc[:n_train].reset_index(drop=True)
mean3, std3 = cv_score(X3, y_train)
results['3. + engineered features (raw kept)'] = (mean3, std3, X3.shape[1])
print(f'Step 3 -- engineered features, raw columns kept:  log-RMSE {mean3:.4f} (+/- {std3:.4f})  [{X3.shape[1]} features]')

Step 3 -- engineered features, raw columns kept:  log-RMSE 0.1156 (+/- 0.0083)  [214 features]


**Reading it:** exactly as predicted -- log-RMSE is unchanged (0.1156 vs.
0.1154), because we handed Ridge information it already had, in a different
shape. The features aren't *wrong*, they're *redundant* while the raw
columns they're built from are still sitting right next to them in the
model.

### Step 3b: same features, but now drop the raw columns they replace

If the aggregated feature is meant to *replace* its parts rather than sit
alongside them, drop the parts. This isn't about giving Ridge new
information -- it's about giving it **fewer, less collinear parameters** to
shrink. Five separate porch-area columns (several of them near-zero for most
houses) are noisier to regularize than one `TotalPorchSF`; the same logic
applies to the basement/floor columns collapsing into `TotalSF` and the four
bathroom-count columns collapsing into `TotalBath`.

In [6]:
redundant_raw_cols = [
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',            # replaced by TotalSF
    'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'WoodDeckSF',  # replaced by TotalPorchSF
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath',  # replaced by TotalBath
]
step3b = step3.drop(columns=redundant_raw_cols)

categorical_cols_3b = step3b.select_dtypes(exclude=[np.number]).columns
step3b_encoded = pd.get_dummies(step3b, columns=list(categorical_cols_3b), drop_first=True)
X3b = step3b_encoded.iloc[:n_train].reset_index(drop=True)
mean3b, std3b = cv_score(X3b, y_train)
results['3b. + engineered features (raw dropped)'] = (mean3b, std3b, X3b.shape[1])
print(f'Step 3b -- engineered features, raw columns dropped:  log-RMSE {mean3b:.4f} (+/- {std3b:.4f})  [{X3b.shape[1]} features]')

Step 3b -- engineered features, raw columns dropped:  log-RMSE 0.1153 (+/- 0.0081)  [202 features]


**Reading it:** now there's a real, if modest, improvement: log-RMSE **0.1153**
(down from 0.1154), with a tighter fold-to-fold spread (std 0.0081 vs.
0.0091) and 64 fewer features (202 vs. 266) than where we started. The lesson
isn't "feature engineering doesn't work" -- it's that *for a linear model
specifically*, an aggregated feature only earns its keep by replacing
correlated raw columns, not by joining them. (For a tree-based model later,
you'd likely keep both forms -- trees don't get confused by redundant
columns the way collinearity confuses a linear model's individual
coefficients, and the raw columns may still support splits the aggregate
can't.)

## Step 4: tune `alpha` instead of guessing it

The baseline picked `Ridge(alpha=10)` as "a reasonable guess". `RidgeCV`
searches a grid of `alpha` values using the same cross-validation, and picks
whichever minimizes CV error -- turning a guess into a searched value, on the
final (step 3b) feature set.

In [7]:
alphas = np.logspace(-2, 3, 100)
ridge_cv = RidgeCV(alphas=alphas, cv=kf, scoring='neg_root_mean_squared_error')
ridge_cv.fit(X3b, y_train)
best_alpha = ridge_cv.alpha_
print(f'Best alpha found by search: {best_alpha:.2f}  (baseline had guessed 10)')

mean4, std4 = cv_score(X3b, y_train, alpha=best_alpha)
results[f'4. + tuned alpha ({best_alpha:.1f})'] = (mean4, std4, X3b.shape[1])
print(f'Step 4 -- tuned alpha:  log-RMSE {mean4:.4f} (+/- {std4:.4f})  [{X3b.shape[1]} features]')

Best alpha found by search: 21.54  (baseline had guessed 10)
Step 4 -- tuned alpha:  log-RMSE 0.1150 (+/- 0.0083)  [202 features]


**Reading it:** the search lands on `alpha ≈ 19-22` -- about double the
baseline's guess of 10 -- and log-RMSE ticks down to **0.1150**, the best of
every step so far. Not a dramatic jump, but a free one: this cost a grid
search, not a modelling decision or a new assumption about the data.

## Comparing every step side by side

In [8]:
comparison = pd.DataFrame([
    {'step': name, 'log_rmse_mean': m, 'log_rmse_std': s, 'n_features': n}
    for name, (m, s, n) in results.items()
])
comparison

,step,log_rmse_mean,log_rmse_std,n_features
0,0. baseline (repro),0.115408,0.009099,266
1,1. semantic missing-value fix,0.115622,0.009220,258
2,2. + ordinal encoding,0.115486,0.008382,204
3,3. + engineered features (raw kept),0.115589,0.008259,214
4,3b. + engineered features (raw dropped),0.115314,0.008092,202
5,4. + tuned alpha (21.5),0.115002,0.008254,202


**Reading it end to end:** log-RMSE moved from **0.1154 to 0.1150** -- about
a 0.4% relative improvement, using less than half the fold-to-fold noise
band (~0.008) as a sanity check for "is this real". That's a modest number
for a lot of careful work, and that's an honest, useful thing to learn: for a
**linear, regularized** model, correctness fixes and hand-built features
mostly either (a) fix real semantic problems without moving the CV score, or
(b) get absorbed for free by a model that can already represent linear
combinations of its inputs. The clearest win in this notebook (Step 3b) came
not from *adding* information, but from **removing redundant, collinear
columns** once their replacement existed -- reducing what Ridge had to
regularize.

**What would actually move the needle further from here:**
- **Tree-based / gradient-boosted models** (Random Forest, XGBoost,
  LightGBM) -- these can't reconstruct sums or ratios from individual
  splits, so `TotalSF`, `TotalBath`, and similar aggregates tend to matter
  far more for them than they did for Ridge here.
- **Explicit interaction or polynomial terms** (e.g. `OverallQual x
  GrLivArea`) -- unlike a plain sum, an interaction term genuinely isn't
  representable by a linear model as a combination of its separate inputs,
  so it can add real new information to Ridge specifically.
- **Lasso / ElasticNet** -- automatic feature selection instead of manual
  redundant-column removal, which is exactly the judgment call Step 3b made
  by hand.

## Generate a submission from the best pipeline so far

In [9]:
X_train_final = X3b
X_test_final = step3b_encoded.iloc[n_train:].reset_index(drop=True)

model = Ridge(alpha=best_alpha)
model.fit(X_train_final, y_train)

log_preds = model.predict(X_test_final)
preds = np.expm1(log_preds)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
submission.to_csv('../submissions/submission_feature_engineered.csv', index=False)
submission.head()

,Id,SalePrice
0,1461,112998.111394
1,1462,153719.274032
2,1463,177693.028597
3,1464,199190.118180
4,1465,187405.987594
